# Topic Analysis: Synopses of Scholarship

Identifying topics in medieval sermons can be an underwhelming experience, especially when done in isolation from the text of the documents, and especially if you couldn't read the text even if you had it in front of you. So let's practise with semantic models again, but this time let's focus on a more accessible corpus.

For the past twenty years, I have summarized much of the scholarship I've read, and appended tags to these synopses reflecting their subject matter; in other words, we have a corpus of over 800 labelled data points. You can browse these entries at <https://lib.langeslag.org>, but I have also made them available in the repository under `misc/scholarship.csv`.

Since we are working with a Modern English corpus this time, let's forego the stopword list in favour of frequency restrictions on our `gensim` lexicon.

First let's read in the corpus. I have exported my MySQL database in CSV (comma separated values) format, so we can access it either by interpreting the file's delimiters ourselves, or else using the `csv` library. Also, let's strip away unnecessary fields. We'll want to retain the text of each synopsis, but converted from HTML to tokenized plaintext; and let's also set aside author and date as an identifier, as well as those manual thematic tags, so we can use human topic analysis as a benchmark.

In [1]:
import csv,pyLDAvis.gensim_models
from pathlib import Path
from copy import copy,deepcopy
from bs4 import BeautifulSoup
from pprint import pprint
from gensim import corpora
from gensim.models import LsiModel,LdaModel
from nltk.tokenize import RegexpTokenizer
# We'll exclude punctuation from our list of tokens.
# Just be aware that this will do unspeakable things to words with apostrophes:
tokenizer = RegexpTokenizer(r'(?:\w|\d)+')

In [2]:
data = dict()
_csv_path = Path.cwd().parent / 'misc' / 'scholarship.csv'
with open(_csv_path) as csv_file:
    csv_reader = csv.DictReader(csv_file, delimiter=',')

    for row in csv_reader:
        entry = dict()
        ref = row['author'].split(',')[0].split(' ')[0] + row['year'].replace('&ndash;', '_')
        if ref in data:
            ref = ref + 'a'
        soup = BeautifulSoup(row['abstract'], 'html.parser')
        entry['abstract'] = tokenizer.tokenize(soup.get_text().lower())
        entry['tags'] = row['tags'].split(',')
        if len(entry['abstract']) > 0:
            data[ref] = entry

In [3]:
data['Angerer2024']

{'abstract': ['presents',
  'a',
  'new',
  'uv',
  'photograph',
  'settling',
  'two',
  'minor',
  'points',
  'of',
  'contention',
  'in',
  'the',
  'reading',
  'of',
  'hebban',
  'olla',
  'vogala',
  'then',
  'goes',
  'on',
  'to',
  'express',
  'agreement',
  'with',
  'the',
  'new',
  'consensus',
  '468',
  'that',
  'the',
  'poem',
  'contained',
  'elements',
  'both',
  'of',
  'flemish',
  'and',
  'kentish',
  'indeed',
  'goes',
  'so',
  'far',
  'as',
  'to',
  'claim',
  'that',
  'the',
  'scribe',
  'appears',
  'to',
  'capitalise',
  'on',
  'the',
  'similarity',
  'between',
  'old',
  'english',
  'and',
  'old',
  'dutch',
  'to',
  'produce',
  'a',
  'carefully',
  'formed',
  'text',
  'in',
  'a',
  'dutch',
  'english',
  'mischsprache',
  'which',
  'is',
  'intelligible',
  'in',
  'both',
  '473',
  'he',
  'meant',
  'to',
  'produce',
  'a',
  'poem',
  'readable',
  'not',
  'only',
  'in',
  'dutch',
  'but',
  'also',
  'in',
  'english',

Next for our document-term matrix:

In [4]:
corpus = []
for k,v in data.items():
    corpus.append(v['abstract'])
dictionary = corpora.Dictionary(corpus)
dictionary.filter_extremes(no_below=10, no_above=0.15)
dtmatrix = [dictionary.doc2bow(doc) for doc in corpus]

Let's just use LDiA this time around.

Having to determine the number of topics manually is a major drawback on these methods. (There are now experimental methods available that do this automatically.) Let's go for a higher number with these data:

In [5]:
ldia = LdaModel(dtmatrix, num_topics=9, id2word=dictionary)

Again, we _could_ inspect the resulting topics in text form:

In [6]:
pprint(ldia.print_topics())

[(0,
  '0.004*"world" + 0.004*"winter" + 0.003*"germanic" + 0.003*"saga" + '
  '0.003*"beowulf" + 0.003*"man" + 0.003*"god" + 0.003*"alfred" + '
  '0.003*"summer" + 0.002*"church"'),
 (1,
  '0.005*"ælfric" + 0.003*"winter" + 0.003*"saga" + 0.003*"women" + '
  '0.003*"saxon" + 0.003*"anglo" + 0.003*"poem" + 0.003*"dream" + '
  '0.003*"gísli" + 0.003*"poetry"'),
 (2,
  '0.007*"ælfric" + 0.004*"world" + 0.003*"genesis" + 0.003*"oe" + '
  '0.003*"winter" + 0.003*"natural" + 0.003*"year" + 0.003*"church" + '
  '0.002*"poem" + 0.002*"bwf"'),
 (3,
  '0.007*"ælfric" + 0.003*"saxon" + 0.003*"anglo" + 0.003*"saga" + '
  '0.003*"winter" + 0.003*"alfred" + 0.003*"beowulf" + 0.003*"poetry" + '
  '0.002*"material" + 0.002*"long"'),
 (4,
  '0.006*"winter" + 0.005*"world" + 0.004*"seasons" + 0.004*"word" + '
  '0.004*"germanic" + 0.004*"words" + 0.003*"beowulf" + 0.003*"season" + '
  '0.003*"natural" + 0.003*"long"'),
 (5,
  '0.008*"ælfric" + 0.005*"beowulf" + 0.003*"verse" + 0.003*"bede" + '
  '0.003

But the visualization is more user-friendly:

In [7]:
%matplotlib inline
vis = pyLDAvis.gensim_models.prepare(ldia, dtmatrix, dictionary)
pyLDAvis.enable_notebook()
pyLDAvis.display(vis)

So now how can we link these analyses back to the original documents?

The document-term vectors are still in the order in which we loaded them, so we can reassociate them with their source documents by index:

In [8]:
def classify(document):
    if isinstance(document, int):
        ref = list(data)[document]
    else:
        ref = copy(document)
        document = list(data).index(ref)
    abstract_incipit = ' '.join(data[ref]['abstract'][:40])
    this_ldia = ldia[dtmatrix[document]]
    sorted_result = sorted(this_ldia, key=lambda x: x[1], reverse=True)
    print(f'{ref}: topic {sorted_result[0][0]} ("{abstract_incipit}. . .")')

In [9]:
classify(4)

Árni1990: topic 0 ("gói góa the latter is a late seventeenth century spelling began on a sunday in the course of our february 18 24 gregorian 8 24 late julian the name occurs in many early mss and together with gormánuðr and þorri. . .")


In [10]:
classify("Granlund1955")

Granlund1955: topic 0 ("southern sweden used to count in weeks rather than months and days from 25 march to 36 june on the julian calendar i e during agricultural spring these weeks were counted backwards each week was associated with specific agricultural tasks. . .")


Now let's group together our documents by topic again:

In [11]:
all_topics = ldia.print_topics()
docs_per_topic = [[] for _ in all_topics]
for doc_id, doc_bow in enumerate(dtmatrix):
    doc_topics = ldia.get_document_topics(doc_bow)
    for topic_id, score in doc_topics:
        docs_per_topic[topic_id].append((doc_id, score))

docs_per_topic_by_score = deepcopy(docs_per_topic)
for doc_list in docs_per_topic_by_score:
    doc_list.sort(key=lambda id_and_score: id_and_score[1], reverse=True)

In [12]:
print(docs_per_topic_by_score[0])

[(94, 0.9971357), (81, 0.9965243), (556, 0.99454063), (85, 0.9937341), (92, 0.9918371), (114, 0.9910503), (591, 0.9898894), (578, 0.989409), (104, 0.9876441), (133, 0.98564833), (310, 0.98466027), (537, 0.9838231), (745, 0.9828887), (76, 0.9825914), (384, 0.98184204), (659, 0.9802266), (56, 0.978623), (230, 0.97445977), (739, 0.9738267), (161, 0.9721978), (719, 0.97034585), (32, 0.9703407), (376, 0.9670475), (128, 0.96613014), (59, 0.96440953), (272, 0.9644081), (349, 0.96224123), (44, 0.9613107), (147, 0.9576261), (8, 0.9519033), (232, 0.9443874), (347, 0.94067645), (308, 0.93645054), (27, 0.92384094), (361, 0.91912216), (580, 0.91911644), (5, 0.9182429), (720, 0.9110166), (504, 0.9110152), (604, 0.9065803), (87, 0.90115297), (276, 0.9011367), (579, 0.9011307), (401, 0.8887598), (42, 0.8819384), (127, 0.87985104), (360, 0.87641025), (37, 0.8729247), (211, 0.86035925), (746, 0.8549296), (520, 0.8546602), (47, 0.85435516), (405, 0.84432316), (57, 0.83976823), (121, 0.8232123), (88, 0.81

And now let's recombine that information with labels and incipits:

In [13]:
topic_groups = []
for topic in docs_per_topic_by_score:
    document_data = dict()
    document_refs = [list(data)[k] for (k,v) in topic]
    for ref in document_refs:
        document_data[ref] = ' '.join(data[ref]['abstract'][:40])
    topic_groups.append(document_data)

Now we can finally inspect the topics while getting a decent glimpse of the relevant documents' content:

In [14]:
topic_groups[0]

{'Tille1889': 'opens his book with a defence of tacitus s tripartite germanic year stating that t he tri partition of the germanic year is an unshakable fact despite grimm and others claiming that tacitus misunderstood the germanic concepts the three germanic',
 'Ogilvie1997': 'proxy and historical data may be used to reconstruct medieval climatic fluctuations in england secondary compilations of historical climate data are typically unreliable lamb s study also falls subject to this criticism 112 13 sources should be scrutinised for time',
 'Cusack1998': '5 anglo saxon missions to the continent argues that the germanic peoples being accustomed to the concept of sacred kingship had no difficulty bestowing that same sacral authority on missionaries 121 willibrord s methods in frisia were comparatively hostile overturning',
 'Pearsall1955': 'studies description in sggk in relation to rhetorical precepts matthew of vendome modified ciceronian categories of rhetoric transferring classes of

Alright; now let's see how the statistical topics stack up against human-assigned labels:

In [15]:
counter = 0
label_groups = []
for topic in docs_per_topic:
    label_data = dict()
    label_data['generated'] = ldia.print_topics()[counter][1].replace(' + ', ',')
    label_refs = [list(data)[k] for (k,v) in topic]
    for ref in label_refs:
        label_data[ref] = ','.join(data[ref]['tags'])
    label_groups.append(label_data)
    counter += 1

In [16]:
label_groups[0]

{'generated': '0.004*"world",0.004*"winter",0.003*"germanic",0.003*"saga",0.003*"beowulf",0.003*"man",0.003*"god",0.003*"alfred",0.003*"summer",0.002*"church"',
 'Alamichel1998': 'old english,me,seasons,lyric,chaucer',
 'Anderson1997': 'old english,seasons,taxonomy,calendar,semantics,etymology,beowulf',
 'Andersson1974': 'night,beowulf,warfare,classics',
 'Árni1990': 'ritual,calendar,iceland,old norse,folkloristics',
 'Árni1980': 'ritual,calendar,iceland,old norse,folkloristics',
 'Baker1995': 'edition,old english,computistics,world ages,cosmology',
 'Bakhtin1981': 'theory,time,space,chronotope,landscape,setting',
 'Baume1998': 'warfare,normandy,history,seasons,autumn',
 'Becker1967': 'edition,computistics,isidore,latin',
 'Beckman1934': 'computistics,calendar,old norse,iceland',
 'Beckman1916': 'computistics,old norse,edition,iceland',
 'Buckland1996': 'climate,climate history,climate change,greenland,old norse,social history,anthropology,subsistence',
 'Burton1894': 'old english,natu

Can you guess from this topic what my doctoral dissertation was on?